In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "Processed_527_datapoints.csv"

df = pd.read_csv(RAW_DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (527, 11)


In [2]:
print("Column names:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Column names:
1. Surface Area (m2/g)
2. Total Pore Volume(cm3/g)
3. Micropore Volume (cm3/g)
4. C (%)
5. H (%)
6. N (%)
7. O (%)
8. S (%)
9. Temp (°C)
10. Pressure (bar)
11. CO2 uptake (mmol/g)


In [3]:
print("Data types:")
print(df.dtypes)

Data types:
Surface Area (m2/g)         float64
Total Pore Volume(cm3/g)    float64
Micropore Volume (cm3/g)    float64
C (%)                       float64
H (%)                       float64
N (%)                       float64
O (%)                       float64
S (%)                       float64
Temp (°C)                     int64
Pressure (bar)              float64
CO2 uptake (mmol/g)         float64
dtype: object


In [4]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().sum() / len(df) * 100).round(2)
})

missing_summary

,missing_count,missing_percentage
Surface Area (m2/g),0,0.00
Total Pore Volume(cm3/g),28,5.31
Micropore Volume (cm3/g),210,39.85
C (%),0,0.00
H (%),0,0.00
N (%),0,0.00
O (%),0,0.00
S (%),0,0.00
Temp (°C),0,0.00
Pressure (bar),0,0.00


In [5]:
TARGET_COLUMN = "CO2 uptake (mmol/g)"

print("Target column:", TARGET_COLUMN)
print("Missing target values:", df[TARGET_COLUMN].isna().sum())

Target column: CO2 uptake (mmol/g)
Missing target values: 0


In [6]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 1


In [7]:
df[df.duplicated(keep=False)]

,Surface Area (m2/g),Total Pore Volume(cm3/g),Micropore Volume (cm3/g),C (%),H (%),N (%),O (%),S (%),Temp (°C),Pressure (bar),CO2 uptake (mmol/g)
208,1636.0,0.74,0.68,54.0,3.75,2.69,39.56,0.0,25,1.0,2.86
229,1636.0,0.74,0.68,54.0,3.75,2.69,39.56,0.0,25,1.0,2.86


In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Surface Area (m2/g),527.0,1488.150152,728.611596,2.48,955.500,1420.00,1960.000,3337.00000
Total Pore Volume(cm3/g),499.0,0.800020,0.448172,0.02,0.480,0.69,1.030,3.09000
Micropore Volume (cm3/g),317.0,0.516845,0.272742,0.04,0.310,0.48,0.690,1.29000
C (%),527.0,76.322676,12.591289,40.40,69.830,79.10,85.330,96.96000
H (%),527.0,1.684080,1.585595,0.00,0.000,1.50,2.410,6.01000
N (%),527.0,3.262524,3.712314,0.00,0.715,2.16,4.450,38.15000
O (%),527.0,18.173321,11.632111,0.00,9.920,15.43,22.910,54.00000
S (%),527.0,0.125351,0.930828,0.00,0.000,0.00,0.000,13.20000
Temp (°C),527.0,18.462998,11.009418,0.00,0.000,25.00,25.000,30.00000
Pressure (bar),527.0,0.866636,0.311198,0.10,1.000,1.00,1.000,1.01325


In [9]:
print("Negative surface area:",
      (df["Surface Area (m2/g)"] < 0).sum())

print("Negative total pore volume:",
      (df["Total Pore Volume(cm3/g)"] < 0).sum())

print("Negative micropore volume:",
      (df["Micropore Volume (cm3/g)"] < 0).sum())

print("Negative pressure:",
      (df["Pressure (bar)"] < 0).sum())

print("Negative CO2 uptake:",
      (df["CO2 uptake (mmol/g)"] < 0).sum())

Negative surface area: 0
Negative total pore volume: 0
Negative micropore volume: 0
Negative pressure: 0
Negative CO2 uptake: 0


In [10]:
mpv_greater_than_tpv = (
    df["Micropore Volume (cm3/g)"] >
    df["Total Pore Volume(cm3/g)"]
)

print("Rows where MPV > TPV:", mpv_greater_than_tpv.sum())

Rows where MPV > TPV: 2


In [11]:
element_columns = [
    "C (%)",
    "H (%)",
    "N (%)",
    "O (%)",
    "S (%)"
]

In [12]:
element_sum = df[element_columns].sum(axis=1)

print("Minimum elemental sum:", element_sum.min())
print("Maximum elemental sum:", element_sum.max())

Minimum elemental sum: 84.97
Maximum elemental sum: 104.0


In [13]:
print("Rows with elemental sum > 100:",
      (element_sum > 100).sum())

print("Rows with elemental sum < 0:",
      (element_sum < 0).sum())

Rows with elemental sum > 100: 71
Rows with elemental sum < 0: 0


In [14]:
non_numeric_columns = df.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Non-numeric columns:", non_numeric_columns)

Non-numeric columns: []


In [15]:
infinite_counts = np.isinf(df.select_dtypes(include="number")).sum()

print(infinite_counts)

Surface Area (m2/g)         0
Total Pore Volume(cm3/g)    0
Micropore Volume (cm3/g)    0
C (%)                       0
H (%)                       0
N (%)                       0
O (%)                       0
S (%)                       0
Temp (°C)                   0
Pressure (bar)              0
CO2 uptake (mmol/g)         0
dtype: int64


In [16]:
audit_report = {
    "rows": len(df),
    "columns": len(df.columns),
    "total_missing_values": int(df.isna().sum().sum()),
    "missing_target_values": int(df[TARGET_COLUMN].isna().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "non_numeric_columns": len(non_numeric_columns),
    "negative_surface_area": int(
        (df["Surface Area (m2/g)"] < 0).sum()
    ),
    "negative_total_pore_volume": int(
        (df["Total Pore Volume(cm3/g)"] < 0).sum()
    ),
    "negative_micropore_volume": int(
        (df["Micropore Volume (cm3/g)"] < 0).sum()
    ),
    "negative_pressure": int(
        (df["Pressure (bar)"] < 0).sum()
    ),
    "negative_target": int(
        (df[TARGET_COLUMN] < 0).sum()
    ),
    "mpv_greater_than_tpv": int(
        mpv_greater_than_tpv.sum()
    ),
    "element_sum_above_100": int(
        (element_sum > 100).sum()
    ),
    "infinite_values": int(
        np.isinf(df.select_dtypes(include="number")).sum().sum()
    )
}

audit_report

{'rows': 527,
 'columns': 11,
 'total_missing_values': 238,
 'missing_target_values': 0,
 'duplicate_rows': 1,
 'non_numeric_columns': 0,
 'negative_surface_area': 0,
 'negative_total_pore_volume': 0,
 'negative_micropore_volume': 0,
 'negative_pressure': 0,
 'negative_target': 0,
 'mpv_greater_than_tpv': 2,
 'element_sum_above_100': 71,
 'infinite_values': 0}